# M5 – chạy thí nghiệm đầy đủ (Kaggle hoặc Colab)
Chạy `scripts/run_experiment.py` trên mẫu 1.200 chuỗi. Đặt `SPLIT`, `MODELS`, `BRANCH` ở ô đầu rồi chạy tuần tự.
Kaggle: thêm dữ liệu cuộc thi *M5 Forecasting – Accuracy* vào Input và bật Internet.
Colab: đặt `DATA_DIR` tới thư mục Drive chứa 5 CSV gốc. Không dán token vào notebook.

In [ ]:
import os, pathlib, subprocess, sys
SPLIT = "test"      # "validation" hoặc "test" (test cần tag config-freeze)
MODELS = "seasonal_naive linear_regression ets croston lightgbm xgboost"
BRANCH = "m5-end-to-end"
REPO = "https://github.com/haiphamt/IS403.R11-M5-Competition.git"
ON_KAGGLE = os.path.exists("/kaggle/working")
WORK = pathlib.Path("/kaggle/working" if ON_KAGGLE else "/content")
DATA_DIR = None if ON_KAGGLE else "/content/drive/MyDrive/M5"   # Kaggle: bộ đọc tự tìm trong /kaggle/input

In [ ]:
if not ON_KAGGLE:
    from google.colab import drive
    drive.mount("/content/drive")
import shutil
os.chdir(WORK)  # tránh đứng trong thư mục repo sắp bị xóa
if (WORK / "repo").exists():
    shutil.rmtree(WORK / "repo")  # luôn clone mới để khớp nhánh/tag
if True:
    subprocess.run(["git", "clone", "--branch", BRANCH, REPO, str(WORK / "repo")], check=True)
os.chdir(WORK / "repo")
print(subprocess.run(["git", "log", "-1", "--format=%H %s"], capture_output=True, text=True).stdout)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "statsforecast", "catboost", "lightgbm", "xgboost", "scikit-learn", "pyarrow", "PyYAML"], check=True)

In [ ]:
import multiprocessing
cpus = multiprocessing.cpu_count(); print("CPU:", cpus)
cmd = [sys.executable, "scripts/run_experiment.py", "--split", SPLIT, "--models", *MODELS.split(),
       "--workers", str(max(1, cpus - 1)), "--n-jobs", str(cpus)]
if DATA_DIR: cmd += ["--data-dir", DATA_DIR]
subprocess.run(cmd, check=True)

In [ ]:
import shutil, glob, pandas as pd
run = sorted(glob.glob("results/runs/*"))[-1]
print(run); print(pd.read_csv(run + "/metrics_summary.csv").round(4).to_string())
shutil.rmtree(run + "/ckpt", ignore_errors=True)
print(shutil.make_archive(str(WORK / f"m5_{SPLIT}_{os.path.basename(run)}"), "zip", run))